# ChronoZ — Macro Dataset Audit (v0.1)

This notebook audits the monthly macro datasets under `data/derived/iterations/` before they are used as features in a predictive model. It focuses on schema, timestamp coverage, missingness, duplicate dates, numeric integrity, category/series feature structure, and potential leakage risks.

**Important:** the tracked files currently use `_template.csv` names. The notebook discovers files by pattern and flags templates/empty numeric content. It does not silently treat missing template values as valid observations, and it does not train a model.

## Audit principles
- Snapshot date is expected to be monthly and aligned to the 25th.
- Observation/snapshot date is not necessarily the macro release/availability date. A true point-in-time backtest requires release timestamps and, ideally, vintage data.
- Historical normalization fitted on the full sample can leak future information. This notebook flags the risk but cannot repair upstream metrics without source data and metadata.
- Outputs are audit diagnostics, not investment signals.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

# Assumes this notebook lives in repository_root/macro-predictiveModel/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'macro-predictiveModel' else Path.cwd()
DATA_DIR = REPO_ROOT / 'data' / 'derived' / 'iterations'
CATEGORY_PATTERN = 'macro_category_dataset*.csv'
SERIES_PATTERN = 'macro_series_dataset*.csv'

print(f'Repository root: {REPO_ROOT.resolve()}')
print(f'Data directory: {DATA_DIR.resolve()}')
print('Exists:', DATA_DIR.exists())
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Could not find {DATA_DIR}. Open this notebook from the repository, '
        'or adjust REPO_ROOT to the local clone root.'
    )

## 1. Discover source files

The repository contains a generator at `scripts/generate_macro_datasets.py` and cleaned source indicators under `data/cleaned/`. This cell **prefers populated output files** (`macro_category_dataset_v0.1.csv`, `macro_series_dataset_v0.1.csv`) over `*_template.csv`. If outputs do not exist locally, it runs the existing generator to create them from the repository's cleaned indicators. It never treats empty templates as populated data.

In [ ]:
category_candidates = sorted(DATA_DIR.glob(CATEGORY_PATTERN))
series_candidates = sorted(DATA_DIR.glob(SERIES_PATTERN))

# Prefer generated datasets (macro_*_dataset_v0.1.csv) over empty *_template.csv files.
category_outputs = [p for p in category_candidates if 'template' not in p.stem.lower()]
series_outputs = [p for p in series_candidates if 'template' not in p.stem.lower()]

print('Category candidates:', [p.name for p in category_candidates])
print('Series candidates:', [p.name for p in series_candidates])

CATEGORY_FILE = category_outputs[-1] if category_outputs else None
SERIES_FILE = series_outputs[-1] if series_outputs else None

# If outputs have not yet been generated locally, use the existing repo generator
# and cleaned indicator CSVs; do not plot the empty templates.
if CATEGORY_FILE is None or SERIES_FILE is None:
    generator = REPO_ROOT / 'scripts' / 'generate_macro_datasets.py'
    if generator.exists():
        import subprocess, sys
        print('Generated macro outputs not found. Running the repository generator...')
        subprocess.run([sys.executable, str(generator)], cwd=REPO_ROOT, check=True)
        category_candidates = sorted(DATA_DIR.glob(CATEGORY_PATTERN))
        series_candidates = sorted(DATA_DIR.glob(SERIES_PATTERN))
        category_outputs = [p for p in category_candidates if 'template' not in p.stem.lower()]
        series_outputs = [p for p in series_candidates if 'template' not in p.stem.lower()]
        CATEGORY_FILE = category_outputs[-1] if category_outputs else None
        SERIES_FILE = series_outputs[-1] if series_outputs else None

print('Selected category file:', CATEGORY_FILE.name if CATEGORY_FILE else 'NOT FOUND')
print('Selected series file:', SERIES_FILE.name if SERIES_FILE else 'NOT FOUND')

if CATEGORY_FILE is None or SERIES_FILE is None:
    raise FileNotFoundError(
        'Could not find populated macro datasets or generate them from scripts/generate_macro_datasets.py. '
        'Check that the repository has data/cleaned and data/cleaned/series_metadata.csv.'
    )

## 2. Load and inspect schemas

The audit normalizes column names only for diagnosis; it does not rename or overwrite source data. It expects one date column named `snapshot_date` (category file) or a compatible date column. Data rows are preserved as loaded.

In [ ]:
category_raw = pd.read_csv(CATEGORY_FILE)
series_raw = pd.read_csv(SERIES_FILE)

display(pd.DataFrame({
    'dataset': ['macro_category', 'macro_series'],
    'file': [CATEGORY_FILE.name, SERIES_FILE.name],
    'rows': [len(category_raw), len(series_raw)],
    'columns': [len(category_raw.columns), len(series_raw.columns)],
    'duplicate_column_names': [category_raw.columns.duplicated().sum(), series_raw.columns.duplicated().sum()],
}))

print('Category columns:')
display(pd.DataFrame({'column': category_raw.columns, 'dtype': category_raw.dtypes.astype(str).values}))
print('Series columns:')
display(pd.DataFrame({'column': series_raw.columns, 'dtype': series_raw.dtypes.astype(str).values}))

display(category_raw.head())
display(series_raw.head())

## 3. Date and row integrity

Checks invalid/missing dates, duplicate snapshots, sorting, expected month-end-ish coverage (day 25), missing calendar months, and duplicate records. Dates on the 25th are expected but not automatically shifted if weekend/holiday conventions differ.

In [ ]:
def date_audit(df: pd.DataFrame, dataset_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    date_col = next((c for c in ['snapshot_date', 'date', 'Date', 'DATE'] if c in df.columns), None)
    if date_col is None:
        return (pd.DataFrame([{'dataset': dataset_name, 'check': 'date column', 'status': 'FAIL', 'detail': 'No recognized date column'}]), pd.DataFrame())

    parsed = pd.to_datetime(df[date_col], errors='coerce')
    issues = []
    issues.append({'dataset': dataset_name, 'check': 'unparseable/missing dates', 'status': 'PASS' if parsed.notna().all() else 'FAIL', 'detail': f'{parsed.isna().sum()} of {len(df)} rows'})
    duplicates = int(parsed.duplicated(keep=False).sum())
    issues.append({'dataset': dataset_name, 'check': 'duplicate date rows', 'status': 'PASS' if duplicates == 0 else 'WARN', 'detail': f'{duplicates} rows share duplicated dates'})
    wrong_day = int(((parsed.dt.day != 25) & parsed.notna()).sum())
    issues.append({'dataset': dataset_name, 'check': 'snapshot day is 25', 'status': 'PASS' if wrong_day == 0 else 'WARN', 'detail': f'{wrong_day} rows are not dated on day 25'})
    sorted_ok = parsed.dropna().is_monotonic_increasing
    issues.append({'dataset': dataset_name, 'check': 'chronological sorting', 'status': 'PASS' if sorted_ok else 'WARN', 'detail': 'Dates are nondecreasing' if sorted_ok else 'Dates are not sorted ascending'})

    valid = parsed.dropna()
    if not valid.empty:
        periods = valid.dt.to_period('M')
        present = set(periods.astype(str))
        expected = pd.period_range(periods.min(), periods.max(), freq='M')
        missing_months = [str(p) for p in expected if str(p) not in present]
    else:
        missing_months = []
    issues.append({'dataset': dataset_name, 'check': 'calendar month coverage', 'status': 'PASS' if not missing_months else 'WARN', 'detail': f'{len(missing_months)} missing months' + (f': {missing_months[:12]}' if missing_months else '')})

    result = df.copy()
    result['_parsed_snapshot_date'] = parsed
    return pd.DataFrame(issues), result

category_date_checks, category_dated = date_audit(category_raw, 'macro_category')
series_date_checks, series_dated = date_audit(series_raw, 'macro_series')
date_checks = pd.concat([category_date_checks, series_date_checks], ignore_index=True)
display(date_checks)

## 4. Missingness and numeric validity

This section reports column-level missingness, coerces numeric-looking feature columns for diagnostic summaries only, and identifies all-empty or constant columns. It does not impute data. Empty template cells are expected to be flagged.

In [ ]:
def numeric_audit(df: pd.DataFrame, dataset_name: str, date_candidates=('snapshot_date', 'date', 'Date', 'DATE')) -> tuple[pd.DataFrame, pd.DataFrame]:
    date_col = next((c for c in date_candidates if c in df.columns), None)
    feature_cols = [c for c in df.columns if c != date_col]
    rows = []
    numeric = pd.DataFrame(index=df.index)
    for col in feature_cols:
        values = pd.to_numeric(df[col], errors='coerce')
        non_null_original = df[col].notna().sum()
        rows.append({
            'dataset': dataset_name,
            'column': col,
            'non_null_original': int(non_null_original),
            'numeric_non_null': int(values.notna().sum()),
            'missing_or_non_numeric': int(values.isna().sum()),
            'missing_pct': float(values.isna().mean() * 100) if len(values) else np.nan,
            'unique_numeric_values': int(values.nunique(dropna=True)),
            'constant_or_empty': bool(values.nunique(dropna=True) <= 1),
            'infinite_values': int(np.isinf(values.dropna()).sum()),
            'min': values.min(),
            'median': values.median(),
            'max': values.max(),
            'mean': values.mean(),
            'std': values.std(ddof=1),
        })
        numeric[col] = values
    return pd.DataFrame(rows), numeric

category_numeric_summary, category_numeric = numeric_audit(category_raw, 'macro_category')
series_numeric_summary, series_numeric = numeric_audit(series_raw, 'macro_series')
numeric_summary = pd.concat([category_numeric_summary, series_numeric_summary], ignore_index=True)

display(numeric_summary.sort_values(['dataset', 'missing_pct'], ascending=[True, False]))
display(numeric_summary.groupby('dataset').agg(
    feature_columns=('column', 'count'),
    fully_empty_columns=('numeric_non_null', lambda x: int((x == 0).sum())),
    constant_or_empty_columns=('constant_or_empty', 'sum'),
    columns_with_infinite_values=('infinite_values', lambda x: int((x > 0).sum()))
).reset_index())

## 5. Feature-pair consistency: spread and spread × ROC

For the series dataset, each `*_spread_x_roc4` column should have a corresponding `*_spread` column. This is a structural check only; verifying the formula numerically requires the underlying ROC columns or source indicator history.

In [ ]:
def interaction_pair_audit(columns):
    columns = set(columns)
    interactions = sorted(c for c in columns if c.endswith('_spread_x_roc4'))
    rows = []
    for interaction in interactions:
        spread = interaction.removesuffix('_x_roc4')
        # For e.g. INDPRO_spread_x_roc4 -> INDPRO_spread
        rows.append({'interaction_column': interaction, 'expected_spread_column': spread, 'spread_exists': spread in columns})
    return pd.DataFrame(rows)

display(interaction_pair_audit(series_raw.columns))
display(interaction_pair_audit(category_raw.columns))

series_spread_cols = [c for c in series_raw.columns if c.endswith('_spread')]
category_spread_cols = [c for c in category_raw.columns if c.endswith('_spread') and not c.endswith('_spread_x_roc4')]
print(f'Series spread columns: {len(series_spread_cols)}')
print(f'Category spread columns: {len(category_spread_cols)}')
print(f'Series interaction columns: {sum(c.endswith("_spread_x_roc4") for c in series_raw.columns)}')
print(f'Category interaction columns: {sum(c.endswith("_spread_x_roc4") for c in category_raw.columns)}')

## 6. Cross-dataset date alignment

Both datasets should cover compatible monthly snapshot dates. This section detects missing dates, extra dates, and mismatched duplicate-date structure. It does not fill rows or values.

In [ ]:
def parsed_date_set(df):
    col = next((c for c in ['snapshot_date', 'date', 'Date', 'DATE'] if c in df.columns), None)
    if col is None:
        return set()
    return set(pd.to_datetime(df[col], errors='coerce').dropna())

category_dates = parsed_date_set(category_raw)
series_dates = parsed_date_set(series_raw)
alignment = pd.DataFrame([
    {'metric': 'unique category dates', 'value': len(category_dates)},
    {'metric': 'unique series dates', 'value': len(series_dates)},
    {'metric': 'dates common to both', 'value': len(category_dates & series_dates)},
    {'metric': 'dates only in category', 'value': len(category_dates - series_dates)},
    {'metric': 'dates only in series', 'value': len(series_dates - category_dates)},
])
display(alignment)
print('Category-only examples:', sorted(category_dates - series_dates)[:10])
print('Series-only examples:', sorted(series_dates - category_dates)[:10])

## 7. Visual diagnostics

Plots help detect empty features, changing availability, extreme values and broad structural breaks. They are descriptive and should not be interpreted as evidence of predictability.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
numeric_summary.groupby('dataset')['missing_pct'].mean().plot(kind='bar', ax=ax)
ax.set_title('Average feature missingness by dataset')
ax.set_ylabel('Missing / non-numeric (%)')
ax.set_xlabel('Dataset')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

for name, df in [('macro_category', category_raw), ('macro_series', series_raw)]:
    date_col = next((c for c in ['snapshot_date', 'date', 'Date', 'DATE'] if c in df.columns), None)
    if date_col:
        parsed = pd.to_datetime(df[date_col], errors='coerce')
        num = df.drop(columns=[date_col]).apply(pd.to_numeric, errors='coerce')
        coverage = num.notna().sum(axis=1)
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(parsed, coverage)
        ax.set_title(f'{name}: count of non-missing features per snapshot')
        ax.set_ylabel('Available features')
        ax.set_xlabel('Snapshot date')
        plt.tight_layout()
        plt.show()

        if num.shape[1] > 0:
            fig, ax = plt.subplots(figsize=(12, 4))
            num.isna().mean(axis=0).sort_values(ascending=False).head(20).plot(kind='bar', ax=ax)
            ax.set_title(f'{name}: 20 features with greatest missingness')
            ax.set_ylabel('Missing fraction')
            ax.set_xlabel('Feature')
            plt.xticks(rotation=75, ha='right')
            plt.tight_layout()
            plt.show()

## 8. ML readiness gate

The notebook assigns a transparent readiness status. `BLOCKED` means the current data cannot safely be treated as a training dataset without further work. `CONDITIONAL` means structural checks may pass but point-in-time release/vintage and upstream transformation provenance still need confirmation.

In [ ]:
def dataset_gate(df, numeric_summary_df, dataset_name):
    date_col = next((c for c in ['snapshot_date', 'date', 'Date', 'DATE'] if c in df.columns), None)
    if date_col is None or len(df) == 0:
        return {'dataset': dataset_name, 'gate': 'BLOCKED', 'reason': 'No recognized date column or no rows'}
    feature_cols = [c for c in df.columns if c != date_col]
    if not feature_cols:
        return {'dataset': dataset_name, 'gate': 'BLOCKED', 'reason': 'No feature columns'}
    features = df[feature_cols].apply(pd.to_numeric, errors='coerce')
    if features.notna().sum().sum() == 0:
        return {'dataset': dataset_name, 'gate': 'BLOCKED', 'reason': 'All feature cells are empty or non-numeric (likely template)'}
    if pd.to_datetime(df[date_col], errors='coerce').isna().any():
        return {'dataset': dataset_name, 'gate': 'BLOCKED', 'reason': 'Invalid or missing snapshot dates'}
    if pd.to_datetime(df[date_col], errors='coerce').duplicated().any():
        return {'dataset': dataset_name, 'gate': 'BLOCKED', 'reason': 'Duplicate snapshot dates'}
    return {'dataset': dataset_name, 'gate': 'CONDITIONAL', 'reason': 'Structural checks passed; confirm real-time release/vintage and upstream normalization provenance'}

readiness = pd.DataFrame([
    dataset_gate(category_raw, category_numeric_summary, 'macro_category'),
    dataset_gate(series_raw, series_numeric_summary, 'macro_series'),
])
display(readiness)

print('Readiness gate is intentionally conservative: passing this notebook is not equivalent to a validated backtest.')

## 9. Recommended fixes before ML

1. Replace template CSVs with populated exports and make the source filenames/version explicit.
2. Store `snapshot_date` separately from each indicator's observation period and release/availability timestamp.
3. Document the fitting window used for each benchmark mean/std and verify that historical rows were not normalized using future data.
4. Verify that every `spread_x_roc4` was calculated from its intended spread and ROC definition, including category aggregation order.
5. Decide a single asset and provide its price history separately; this audit notebook intentionally does not invent or fetch asset-price data.
6. Only after the above, construct forward-return targets and use chronological walk-forward validation.

## 10. Overlay plot: macro-category series + polynomial fits

This is a **visual diagnostic**, not a predictive model. It overlays each macro-category `*_spread` time series with a polynomial curve fitted across the available observations. The curve is fitted on the full displayed sample, so it is retrospective and must **not** be used as a point-in-time trading signal or as an ML feature.

- `POLY_DEGREE = 2` gives a quadratic curve; try `3` for a cubic curve.
- Only category `*_spread` columns are plotted (not the `*_spread_x_roc4` interaction features).
- The notebook now selects generated CSV outputs before templates, and runs the existing generator from `data/cleaned/` if outputs are missing. If the generated data still have too few valid points, the cell reports that rather than fabricating a plot.
- The x-axis is calendar time. Polynomial fitting uses a scaled numeric time index for numerical stability.

In [ ]:
POLY_DEGREE = 2  # Change to 3 for a cubic curve

date_col = next((c for c in ['snapshot_date', 'date', 'Date', 'DATE'] if c in category_raw.columns), None)
category_cols = [
    c for c in category_raw.columns
    if c.endswith('_spread') and not c.endswith('_spread_x_roc4')
]

if date_col is None:
    print('Cannot plot polynomial overlay: no recognized date column in category dataset.')
elif not category_cols:
    print('Cannot plot polynomial overlay: no category *_spread columns were found.')
else:
    plot_df = category_raw[[date_col] + category_cols].copy()
    plot_df[date_col] = pd.to_datetime(plot_df[date_col], errors='coerce')
    for col in category_cols:
        plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    plot_df = plot_df.dropna(subset=[date_col]).sort_values(date_col)

    available_cols = [c for c in category_cols if plot_df[c].notna().sum() >= POLY_DEGREE + 1]
    if not available_cols:
        print(
            'No populated category spread has enough numeric observations for the selected polynomial degree. '
            'Populate the category dataset first; no sample data were generated.'
        )
    else:
        fig, ax = plt.subplots(figsize=(15, 7))
        plotted = 0
        for col in available_cols:
            series = plot_df[[date_col, col]].dropna().sort_values(date_col)
            if len(series) < POLY_DEGREE + 1:
                continue

            dates = series[date_col]
            y = series[col].to_numpy(dtype=float)
            # Convert dates to a scaled index in [-1, 1] to improve polynomial conditioning.
            x_days = (dates - dates.min()).dt.total_seconds().to_numpy() / 86400.0
            x_span = x_days.max() - x_days.min()
            if x_span == 0:
                print(f'Skipping {col}: all observations share the same timestamp.')
                continue
            x_scaled = 2 * (x_days / x_span) - 1
            coeffs = np.polyfit(x_scaled, y, deg=POLY_DEGREE)
            y_fit = np.polyval(coeffs, x_scaled)

            line, = ax.plot(dates, y, marker='o', markersize=3, linewidth=1.2, alpha=0.65, label=f'{col} — observed')
            ax.plot(dates, y_fit, linestyle='--', linewidth=2.2, color=line.get_color(), label=f'{col} — degree {POLY_DEGREE} polynomial')
            plotted += 1

        if plotted:
            ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
            ax.set_title(f'Macro-category spreads with polynomial overlay (degree {POLY_DEGREE})')
            ax.set_xlabel('Snapshot date')
            ax.set_ylabel('Category spread')
            ax.grid(True, alpha=0.25)
            ax.legend(ncol=2, fontsize=8)
            fig.tight_layout()
            plt.show()
        else:
            print('No category series could be plotted after validation.')